In [12]:
from google.colab import files
uploaded = files.upload()


Saving Churn_Modelling.csv to Churn_Modelling (1).csv


In [13]:
!pip install xgboost

## 1)Import Libraries

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, silhouette_score)
from xgboost import XGBClassifier
import joblib
import json

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv("Churn_Modelling.csv")
print("Shape:", df.shape)
print(df.head())
print(df.isnull().sum())


Shape: (10000, 14)
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63     

## 2)Data Cleaning

In [15]:

df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

le_geo = LabelEncoder()
le_gender = LabelEncoder()
df['Geography_enc'] = le_geo.fit_transform(df['Geography'])
df['Gender_enc'] = le_gender.fit_transform(df['Gender'])

feature_cols = ['CreditScore', 'Geography_enc', 'Gender_enc', 'Age', 'Tenure',
                 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                 'EstimatedSalary']

X = df[feature_cols]
y = df['Exited']

print("\nChurn distribution:\n", y.value_counts(normalize=True))



Churn distribution:
 Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64


## 3) Train/Test Split + Scaling

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4)Classification Models - RANDOM FOREST & XGBOOST

In [17]:

results = {}

rf = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_split=4,
                             random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

results['Random Forest'] = {
    'accuracy': accuracy_score(y_test, rf_pred),
    'precision': precision_score(y_test, rf_pred),
    'recall': recall_score(y_test, rf_pred),
    'f1': f1_score(y_test, rf_pred),
    'roc_auc': roc_auc_score(y_test, rf_proba)
}

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     scale_pos_weight=scale_pos_weight,
                     random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

results['XGBoost'] = {
    'accuracy': accuracy_score(y_test, xgb_pred),
    'precision': precision_score(y_test, xgb_pred),
    'recall': recall_score(y_test, xgb_pred),
    'f1': f1_score(y_test, xgb_pred),
    'roc_auc': roc_auc_score(y_test, xgb_proba)
}

results_df = pd.DataFrame(results).T
print("\nModel Comparison:\n", results_df)

# Pick best model based on ROC-AUC
best_model_name = results_df['roc_auc'].idxmax()
best_model = rf if best_model_name == 'Random Forest' else xgb
best_pred = rf_pred if best_model_name == 'Random Forest' else xgb_pred
print(f"\nBest Model: {best_model_name}")
print("\nClassification Report:\n", classification_report(y_test, best_pred))



Model Comparison:
                accuracy  precision    recall        f1   roc_auc
Random Forest    0.8405   0.598655  0.656020  0.626026  0.857323
XGBoost          0.8085   0.521352  0.719902  0.604747  0.858885

Best Model: XGBoost

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.83      0.87      1593
           1       0.52      0.72      0.60       407

    accuracy                           0.81      2000
   macro avg       0.72      0.78      0.74      2000
weighted avg       0.84      0.81      0.82      2000



## 5) Feature Importance

In [18]:

importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature Importances:\n", importances)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model comparison bar chart
results_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].plot(
    kind='bar', ax=axes[0], rot=0)
axes[0].set_title('Model Performance Comparison')
axes[0].set_ylabel('Score')
axes[0].legend(loc='lower right', fontsize=8)

# Feature importance
importances.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title(f'Feature Importance ({best_model_name})')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('model_performance.png', bbox_inches='tight')
plt.close()

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, pred) in zip(axes, [('Random Forest', rf_pred), ('XGBoost', xgb_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Stayed', 'Churned'], yticklabels=['Stayed', 'Churned'])
    ax.set_title(f'{name} - Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.close()


Feature Importances:
 NumOfProducts      0.295135
Age                0.159823
IsActiveMember     0.148680
Geography_enc      0.083554
Gender_enc         0.073696
Balance            0.072226
CreditScore        0.047686
EstimatedSalary    0.045535
Tenure             0.041074
HasCrCard          0.032590
dtype: float32


## 6) Customer Segmentation -K-MEANS

In [19]:

cluster_features = ['CreditScore', 'Age', 'Balance', 'NumOfProducts',
                     'EstimatedSalary', 'Tenure']
X_cluster = df[cluster_features]
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

# Elbow method
inertias = []
sil_scores = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, marker='o')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')

axes[1].plot(list(K_range), sil_scores, marker='o', color='orange')
axes[1].set_title('Silhouette Scores')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.savefig('kmeans_selection.png', bbox_inches='tight')
plt.close()

best_k = list(K_range)[np.argmax(sil_scores)]
print(f"\nBest k by silhouette score: {best_k}")

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_cluster_scaled)

cluster_profile = df.groupby('KMeans_Cluster')[cluster_features + ['Exited']].mean()
cluster_profile['Count'] = df['KMeans_Cluster'].value_counts().sort_index()
print("\nK-Means Cluster Profiles:\n", cluster_profile)






Best k by silhouette score: 2

K-Means Cluster Profiles:
                 CreditScore        Age        Balance  NumOfProducts  \
KMeans_Cluster                                                         
0                651.417558  39.119007  123033.820983       1.284926   
1                649.302475  38.649691   12258.162499       1.868634   

                EstimatedSalary    Tenure    Exited  Count  
KMeans_Cluster                                              
0                 100727.333001  4.967058  0.225423   5798  
1                  99211.166604  5.075916  0.173727   4202  


## 7) DBSCAN Clustering

In [20]:

dbscan = DBSCAN(eps=1.2, min_samples=10)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_cluster_scaled)
n_clusters_db = len(set(df['DBSCAN_Cluster'])) - (1 if -1 in df['DBSCAN_Cluster'].values else 0)
n_noise = (df['DBSCAN_Cluster'] == -1).sum()
print(f"\nDBSCAN found {n_clusters_db} clusters and {n_noise} noise points")

dbscan_profile = df.groupby('DBSCAN_Cluster')[cluster_features + ['Exited']].mean()
dbscan_profile['Count'] = df['DBSCAN_Cluster'].value_counts().sort_index()
print("\nDBSCAN Cluster Profiles:\n", dbscan_profile)

# Visualize clusters using PCA for 2D projection
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['KMeans_Cluster'],
                            cmap='tab10', s=10, alpha=0.6)
axes[0].set_title(f'K-Means Clusters (k={best_k}) - PCA Projection')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=df['DBSCAN_Cluster'],
                            cmap='tab10', s=10, alpha=0.6)
axes[1].set_title(f'DBSCAN Clusters - PCA Projection')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')
plt.tight_layout()
plt.savefig('clustering_results.png', bbox_inches='tight')
plt.close()


DBSCAN found 5 clusters and 242 noise points

DBSCAN Cluster Profiles:
                 CreditScore        Age        Balance  NumOfProducts  \
DBSCAN_Cluster                                                         
-1               646.690083  53.322314   80650.845950       2.834711   
 0               649.387020  39.497428   98816.400378       1.000000   
 1               651.977143  37.458681   51473.161688       2.000000   
 2               641.414062  41.367188   64574.832187       3.000000   
 3               711.500000  39.666667  131597.045556       3.000000   
 4               672.875000  31.000000  135761.203750       3.000000   

                EstimatedSalary    Tenure    Exited  Count  
DBSCAN_Cluster                                              
-1                109110.492355  5.392562  0.706612    242  
 0                 99343.995894  4.968540  0.277008   5054  
 1                100463.646826  5.052747  0.075165   4550  
 2                104372.346797  4.664062  0.

## 8)Save Artifacts

In [21]:

df.to_csv('customer_segments_output.csv', index=False)
joblib.dump(best_model, 'best_churn_model.pkl')

summary = {
    'model_results': {k: {m: round(v, 4) for m, v in vals.items()} for k, vals in results.items()},
    'best_model': best_model_name,
    'feature_importance': {k: round(v, 4) for k, v in importances.items()},
    'best_k_kmeans': int(best_k),
    'kmeans_cluster_profile': cluster_profile.round(2).to_dict(),
    'dbscan_n_clusters': int(n_clusters_db),
    'dbscan_n_noise': int(n_noise),
    'dbscan_cluster_profile': dbscan_profile.round(2).to_dict()
}

with open('results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print("\n=== DONE ===")



=== DONE ===
